In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup

In [4]:

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [63]:
system_message = """Você é um assistente útil que pode buscar e resumir notícias da internet.
Quando o usuário pedir para buscar notícias de um site específico, use a função create_brochure.
Você também pode conversar normalmente sobre outros assuntos."""

In [13]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    Uma classe utilitária para representar um site que foi raspado (scrapeado), agora incluindo os links.
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [17]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 '

In [33]:
link_system_prompt = "Você recebe uma lista de links encontrados em uma página da web.\
Você deve decidir quais desses links seriam mais relevantes para incluir em um \
resumo das notícias que encontrara nos links. "
link_system_prompt += "Você deve responder em JSON conforme o exemplo:"
link_system_prompt += """
{
  "links": [
    {"type": "news article", "url": "https://exemplo.com/noticia1"}
  ]
}
Limite a 5 links mais relevantes.
"""

In [37]:
print(link_system_prompt)

Você recebe uma lista de links encontrados em uma página da web.Você deve decidir quais desses links seriam mais relevantes para incluir em um resumo das notícias que encontrara nos links. Você deve responder em JSON conforme o exemplo:
{
  "links": [
    {"type": "news article", "url": "https://exemplo.com/noticia1"}
  ]
}



In [62]:
def get_links_user_prompt(website):
    user_prompt = (
        f"Aqui está uma lista de links do website {website.url}.\n"
        "Por favor, decida quais desses links são relevantes para um resumo de notícias e responda apenas com o URL completo (https) no formato JSON.\n"
        "Inclua somente links relacionados a notícias, artigos ou comunicados — exclua todos os outros.\n\n"
        "Links:\n"
    )
    user_prompt += "\n".join(website.links[:50])
    return user_prompt


In [19]:
print(get_links_user_prompt(ed))

Aqui está uma lista de links do website https://edwarddonner.com.
Por favor, decida quais desses links são relevantes para um resumo de notícias e responda apenas com o URL completo (https) no formato JSON.
Inclua somente links relacionados a notícias, artigos ou comunicados — exclua todos os outros.

Links:
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
ht

In [20]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [21]:
get_links("https://edwarddonner.com")

{'links': [{'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'},
  {'url': 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/'},
  {'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'}]}

In [64]:
def get_all_details(url):
    """Coleta detalhes da página principal e links relevantes"""
    result = "=== Página Principal ===\n"
    result += Website(url).get_contents()
    
    links = get_links(url)
    print(f"Links encontrados: {len(links.get('links', []))}")
    
    for i, link in enumerate(links.get("links", [])[:3]):  # Limita a 3 links
        try:
            result += f"\n\n=== {link.get('type', 'Link')} {i+1} ===\n"
            result += Website(link["url"]).get_contents()
        except Exception as e:
            print(f"Erro ao processar link {link.get('url')}: {e}")
            continue
    
    return result[:8000]  

In [60]:
get_all_details("https://edwarddonner.com")

Found links: {'links': [{'type': 'news article', 'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'}]}


'Página principal:\nWebpage Title:\nHome - Edward Donner\nWebpage Contents:\nHome\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nWe work with groundbreaking, proprietary LLMs verticalized for talent, we’ve\npatented\nour matching model, and our aw

In [30]:
system_prompt = "Você é um assistente que analisa o conteúdo de várias páginas relevantes do site de notícias \
e cria um breve folheto sobre as principais notícias do dia. Responda em markdown. \
Inclua detalhes sobre as notícias."

In [65]:
def create_brochure(url):
    """Cria um resumo de notícias a partir de uma URL"""
    try:
        user_prompt = (
            f"Analise o seguinte conteúdo da web e crie um resumo das principais notícias "
            f"encontradas. Organize em tópicos claros e seja conciso.\n\n"
        )
        user_prompt += get_all_details(url)
        
        response = openai.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Você é um especialista em resumir notícias de forma clara e objetiva."},
                {"role": "user", "content": user_prompt}
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erro ao criar resumo: {str(e)}"


In [26]:
get_brochure_user_prompt("Ed Donner", "https://edwarddonner.com")

Found links: {'links': [{'type': 'news article', 'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'}]}


'Você está analisando uma empresa chamada: Ed Donner\nAqui estão os conteúdos da sua página inicial e de outras páginas relevantes; use essas informações para criar um breve folheto sobre a empresa em formato markdown.\nLanding page:\nWebpage Title:\nHome - Edward Donner\nWebpage Contents:\nHome\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage a

In [48]:
create_brochure("Ed Donner", "https://edwarddonner.com")

Found links: {'links': [{'type': 'news article', 'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/'}, {'type': 'news article', 'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'}]}


'```markdown\n# Folheto da Empresa - Edward Donner\n\n## Introdução\n**Edward Donner** é uma influente figura no campo da inteligência artificial, sendo o co-fundador e CTO da **Nebula.io**, uma startup que aplica tecnologia de IA para ajudar indivíduos a descobrirem seu potencial e trilhar o caminho para suas realizações pessoais. Com uma carreira sólida, Ed também é conhecido por seu trabalho na **untapt**, uma startup de IA adquirida em 2021.\n\n## Sobre Ed Donner\n- **Interesses**: Ed é apaixonado por programação e experimentação com Modelos de Linguagem de Aprendizado (LLMs). Além disso, se dedica à DJing e à produção musical eletrônica amadora.\n- **Educação e Experiência**:\n  - Ex-fundador e CEO da **untapt**.\n  - Co-fundador e CTO na **Nebula.io**, uma plataforma que oferece soluções de AI focadas em recrutamento e gestão de talentos.\n\n## Nebula.io\n- **Missão**: A Nebula.io visa transformar o recrutamento, oferecendo ferramentas que facilitam a fonte, compreensão, engajame

In [74]:
news_function = {
    "name": "create_brochure",
    "description": "Busca notícias de um site específico e cria um resumo delas",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "A URL do site onde buscar as notícias (deve começar com http:// ou https://)",
            },
        },
        "required": ["url"],
        "additionalProperties": False
    }
}

In [75]:
tools = [{"type": "function", "function": news_function}]

In [76]:
def handle_tool_call(message):
    """Processa chamadas de função"""
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    url = arguments.get('url')
    
    print(f"Buscando notícias de: {url}")
    texto = create_brochure(url)
    
    response = {
        "role": "tool",
        "content": json.dumps({"url": url, "resumo": texto}),
        "tool_call_id": tool_call.id
    }
    return response, url

In [77]:
def chat(message, history):
    """Função principal do chat"""
    # Converte histórico do Gradio para formato OpenAI
    messages = [{"role": "system", "content": system_message}]
    
    for h in history:
        if h["role"] == "user":
            messages.append({"role": "user", "content": h["content"]})
        elif h["role"] == "assistant":
            messages.append({"role": "assistant", "content": h["content"]})
    
    messages.append({"role": "user", "content": message})
    
    # Primeira chamada à API
    response = openai.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools
    )
    
    # Verifica se precisa chamar uma função
    if response.choices[0].finish_reason == "tool_calls":
        message_with_tool = response.choices[0].message
        tool_response, url = handle_tool_call(message_with_tool)
        
        # Adiciona a mensagem da ferramenta e a resposta ao histórico
        messages.append({
            "role": "assistant",
            "content": message_with_tool.content,
            "tool_calls": message_with_tool.tool_calls
        })
        messages.append(tool_response)
        
        # Segunda chamada para gerar resposta final
        response = openai.chat.completions.create(
            model=MODEL, 
            messages=messages
        )
    
    return response.choices[0].message.content

In [36]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
